In [3]:
!pip install -q langchain langchain-groq gradio

In [4]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [10]:
"""
Animal Facts Translator — Gradio UI
--------------------------------------
Generates animal facts via LangChain + Groq, then translates them into a
selected language (English, Urdu, or Arabic) using quick-select buttons.

Setup:
    pip install gradio langchain langchain-groq langchain-core

    Set your Groq API key as an environment variable before running:
        export GROQ_API_KEY="your_key_here"      (macOS/Linux)
        setx GROQ_API_KEY "your_key_here"         (Windows)

Run:
    python animal_facts_translator.py
"""

import os
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ---------------------------------------------------------------------------
# LangChain setup
# ---------------------------------------------------------------------------

MODEL_NAME = "openai/gpt-oss-120b"

animal_facts_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You like telling facts and you tell facts about {animal}."),
        ("human", "Tell me {count} facts."),
    ]
)

translation_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a translator and convert the provided text into {language}."),
        ("human", "Translate the following text to {language}: {text}"),
    ]
)

_model = None


def get_model():
    """Lazily build the model so a missing API key doesn't crash app startup."""
    global _model
    if _model is None:
        _model = ChatGroq(
            model=MODEL_NAME,
            groq_api_key=os.getenv("GROQ_API_KEY"),
        )
    return _model


# Languages that skip the translation step entirely (facts are generated in English)
NO_TRANSLATION_NEEDED = {"English"}


# ---------------------------------------------------------------------------
# Core inference function
# ---------------------------------------------------------------------------

def translate_facts(animal: str, count, language: str):
    if not animal or not animal.strip():
        return "⚠️ Please enter an animal name."

    if not language:
        return "⚠️ Please choose a language (English, Urdu, or Arabic)."

    if not os.getenv("GROQ_API_KEY"):
        return (
            "⚠️ No GROQ_API_KEY found in your environment.\n\n"
            "Set it with:\n"
            "  export GROQ_API_KEY=\"your_key_here\"   (macOS/Linux)\n"
            "  setx GROQ_API_KEY \"your_key_here\"      (Windows)\n"
            "then restart the app."
        )

    try:
        model = get_model()
        parser = StrOutputParser()

        facts_chain = animal_facts_template | model | parser
        facts = facts_chain.invoke({
            "animal": animal.strip(),
            "count": int(count),
        })

        if language in NO_TRANSLATION_NEEDED:
            return facts

        translation_chain = translation_template | model | parser
        translated = translation_chain.invoke({
            "text": facts,
            "language": language,
        })
        return translated

    except Exception as e:
        return f"❌ An error occurred:\n\n{e}"


def clear_fields():
    return "", 3, "English", ""


# ---------------------------------------------------------------------------
# Gradio UI
# ---------------------------------------------------------------------------

CUSTOM_CSS = """
.gradio-container {max-width: 1100px !important; margin: auto;}
footer {visibility: hidden}

#header-banner {
    background: linear-gradient(135deg, #059669 0%, #0d9488 50%, #0891b2 100%);
    border-radius: 16px;
    padding: 24px 26px;
    margin-bottom: 18px;
    box-shadow: 0 8px 24px rgba(5, 150, 105, 0.25);
}
#header-banner h1 {
    color: #ffffff !important;
    font-size: 1.6rem;
    margin: 0 0 6px 0;
    text-align: center;
}
#header-banner p {
    color: #d1fae5 !important;
    text-align: center;
    margin: 0;
    font-size: 0.92rem;
}

#input-card, #output-card {
    background: var(--block-background-fill);
    border: 1px solid var(--border-color-primary);
    border-radius: 14px;
    padding: 18px 20px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.05);
}

/* Language quick-select buttons */
#lang-row {gap: 8px !important; flex-wrap: nowrap !important;}
.lang-btn {
    border-radius: 10px !important;
    font-weight: 600 !important;
    font-size: 0.85rem !important;
    padding: 8px 4px !important;
    min-width: 0 !important;
    border: 2px solid var(--border-color-primary) !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
    white-space: nowrap !important;
}
.lang-btn:hover {
    transform: translateY(-2px);
}
.lang-btn-active {
    background: linear-gradient(135deg, #059669, #0d9488) !important;
    color: white !important;
    border: 2px solid transparent !important;
    box-shadow: 0 4px 14px rgba(13, 148, 136, 0.35) !important;
}

#generate-btn {
    background: linear-gradient(135deg, #059669, #0891b2) !important;
    color: white !important;
    border: none !important;
    font-weight: 600 !important;
    font-size: 1.05rem !important;
    border-radius: 10px !important;
    box-shadow: 0 4px 14px rgba(8, 145, 178, 0.35) !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
}
#generate-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 18px rgba(8, 145, 178, 0.45) !important;
}

#clear-btn {
    border-radius: 10px !important;
    font-weight: 500 !important;
    border: 1px solid var(--border-color-primary) !important;
}

#output-box textarea {
    font-size: 1.05rem !important;
    line-height: 1.8 !important;
}

#footer-note {
    text-align: center;
    color: #9ca3af;
    font-size: 0.85rem;
    margin-top: 18px;
}
"""

THEME = gr.themes.Soft(
    primary_hue="emerald",
    secondary_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui"],
)

with gr.Blocks(title="Animal Facts Translator") as demo:

    # Holds the currently selected language
    language_state = gr.State("English")

    with gr.Row(equal_height=True):
        with gr.Column(scale=1, min_width=380, elem_id="input-card"):
            gr.HTML(
                """
                <div id="header-banner">
                    <h1>🌍 Animal Facts Translator</h1>
                    <p>Generate animal facts and translate them instantly — powered by LangChain + Groq</p>
                </div>
                """
            )

            gr.Markdown("### 🔍 Ask a Question")

            animal_input = gr.Textbox(
                label="Animal",
                placeholder="e.g. Cat, elephant, octopus…",
                value="Cat",
            )

            count_slider = gr.Slider(
                minimum=1,
                maximum=10,
                value=3,
                step=1,
                label="Number of facts",
            )

            gr.Markdown("**🌐 Translate to:**")
            with gr.Row(elem_id="lang-row"):
                en_btn = gr.Button("🇬🇧 English", elem_classes=["lang-btn", "lang-btn-active"], scale=1, min_width=80)
                ur_btn = gr.Button("🇵🇰 Urdu", elem_classes=["lang-btn"], scale=1, min_width=80)
                ar_btn = gr.Button("🇸🇦 Arabic", elem_classes=["lang-btn"], scale=1, min_width=80)

            selected_lang_display = gr.Markdown("Selected language & Press Generate Button")

            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear", elem_id="clear-btn", scale=1)
                generate_btn = gr.Button(
                    "✨ Generate & Translate", elem_id="generate-btn", variant="primary", scale=2
                )

            gr.Examples(
                examples=[
                    ["elephant", 3],
                    ["octopus", 5],
                    ["platypus", 4],
                    ["snow leopard", 2],
                ],
                inputs=[animal_input, count_slider],
                label="Try an example",
            )

        with gr.Column(scale=1, min_width=380, elem_id="output-card"):
            gr.Markdown("### 📋 Results")
            output_box = gr.Textbox(
                label="",
                lines=22,
                interactive=False,
                placeholder="Your translated facts will appear here...",
                elem_id="output-box",
                show_label=False,
                rtl=False,
            )

    gr.HTML(f"<div id='footer-note'>Powered by LangChain + Groq ({MODEL_NAME})</div>")

    # ---- Language button handlers -----------------------------------------

    def select_english():
        return (
            "English",
            "Selected language & Press Generate Button",
            gr.update(elem_classes=["lang-btn", "lang-btn-active"]),
            gr.update(elem_classes=["lang-btn"]),
            gr.update(elem_classes=["lang-btn"]),
            gr.update(rtl=False, text_align="left"),
        )

    def select_urdu():
        return (
            "Urdu",
            "Selected language & Press Generate Button",
            gr.update(elem_classes=["lang-btn"]),
            gr.update(elem_classes=["lang-btn", "lang-btn-active"]),
            gr.update(elem_classes=["lang-btn"]),
            gr.update(rtl=True, text_align="right"),
        )

    def select_arabic():
        return (
            "Arabic",
            "Selected language & Press Generate Button",
            gr.update(elem_classes=["lang-btn"]),
            gr.update(elem_classes=["lang-btn"]),
            gr.update(elem_classes=["lang-btn", "lang-btn-active"]),
            gr.update(rtl=True, text_align="right"),
        )

    en_btn.click(
        fn=select_english,
        inputs=[],
        outputs=[language_state, selected_lang_display, en_btn, ur_btn, ar_btn, output_box],
    )
    ur_btn.click(
        fn=select_urdu,
        inputs=[],
        outputs=[language_state, selected_lang_display, en_btn, ur_btn, ar_btn, output_box],
    )
    ar_btn.click(
        fn=select_arabic,
        inputs=[],
        outputs=[language_state, selected_lang_display, en_btn, ur_btn, ar_btn, output_box],
    )

    # ---- Generate / Clear ---------------------------------------------------

    generate_btn.click(
        fn=translate_facts,
        inputs=[animal_input, count_slider, language_state],
        outputs=output_box,
    )

    animal_input.submit(
        fn=translate_facts,
        inputs=[animal_input, count_slider, language_state],
        outputs=output_box,
    )

    def clear_all():
        return (
            "",
            3,
            gr.update(value="", rtl=False, text_align="left"),
            "English",
            "Selected language & Press Generate Button",
            gr.update(elem_classes=["lang-btn", "lang-btn-active"]),
            gr.update(elem_classes=["lang-btn"]),
            gr.update(elem_classes=["lang-btn"]),
        )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[
            animal_input,
            count_slider,
            output_box,
            language_state,
            selected_lang_display,
            en_btn,
            ur_btn,
            ar_btn,
        ],
    )


if __name__ == "__main__":
    demo.launch(theme=THEME, css=CUSTOM_CSS)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://13af78dae7dc15de55.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
